# Trust-Aware Collaborative Filtering — EXP8
This notebook implements a Trust-Aware Collaborative Filtering recommender:
- Builds ratings and trust matrices
- Predicts ratings using trusted users' ratings (fallback to user avg)
- Recommends top items for a user

In [1]:
import pandas as pd
import numpy as np

# Provided data
ratings_data = {
    'user': [0, 0, 1, 1, 2, 3, 4, 4, 5, 6, 7, 7, 8, 8, 9, 9],
    'item': [0, 1, 0, 2, 2, 1, 3, 4, 0, 1, 2, 3, 4, 0, 1, 2],
    'rating': [4, 5, 5, 3, 2, 4, 4, 2, 5, 4, 3, 2, 5, 4, 4, 5]
}
trust_data = {
    'user': [0, 1, 1, 2, 3, 4, 5, 6, 7, 8],
    'trusted_user': [1, 0, 2, 3, 4, 5, 6, 7, 8, 9]
}

df_ratings = pd.DataFrame(ratings_data)
df_trust = pd.DataFrame(trust_data)

df_ratings.head()

,user,item,rating
0,0,0,4
1,0,1,5
2,1,0,5
3,1,2,3
4,2,2,2


In [2]:
# All users and items (ensure full index)
users = sorted(df_ratings['user'].unique())
items = sorted(df_ratings['item'].unique())

R = df_ratings.pivot(index='user', columns='item', values='rating')
R = R.reindex(index=users, columns=items)  # ensure consistent shape
display(R)

# Precompute user average ratings (skip NaN)
user_avg = R.mean(axis=1, skipna=True)
global_avg = R.stack().mean()
print("User averages:\n", user_avg.to_dict())
print("Global average:", global_avg)

item,0,1,2,3,4
user,,,,,
0,4.0,5.0,NaN,NaN,NaN
1,5.0,NaN,3.0,NaN,NaN
2,NaN,NaN,2.0,NaN,NaN
3,NaN,4.0,NaN,NaN,NaN
4,NaN,NaN,NaN,4.0,2.0
5,5.0,NaN,NaN,NaN,NaN
6,NaN,4.0,NaN,NaN,NaN
7,NaN,NaN,3.0,2.0,NaN
8,4.0,NaN,NaN,NaN,5.0


User averages:
 {0: 4.5, 1: 4.0, 2: 2.0, 3: 4.0, 4: 3.0, 5: 5.0, 6: 4.0, 7: 2.5, 8: 4.5, 9: 4.5}
Global average: 3.8125


In [3]:
T = pd.DataFrame(0, index=users, columns=users, dtype=int)
for _, row in df_trust.iterrows():
    u = row['user']
    v = row['trusted_user']
    if u in T.index and v in T.columns:
        T.at[u, v] = 1

display(T)

,0,1,2,3,4,5,6,7,8,9
0,0,1,0,0,0,0,0,0,0,0
1,1,0,1,0,0,0,0,0,0,0
2,0,0,0,1,0,0,0,0,0,0
3,0,0,0,0,1,0,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0
5,0,0,0,0,0,0,1,0,0,0
6,0,0,0,0,0,0,0,1,0,0
7,0,0,0,0,0,0,0,0,1,0
8,0,0,0,0,0,0,0,0,0,1
9,0,0,0,0,0,0,0,0,0,0


In [4]:
def predict_rating(user, item, R, T, user_avg, global_avg):
    """
    Predict rating for (user, item) using trust-aware weighted average.
    - Use ratings from trusted users for the item (equal weights).
    - If no trusted rating available, fallback to user's average.
    - If user has no ratings, fallback to global average.
    """
    if user not in R.index or item not in R.columns:
        return global_avg

    # Trusted users who the target user trusts
    trusted_users = T.loc[user]
    trusted_users = trusted_users[trusted_users == 1].index.tolist()

    # Collect ratings by trusted users for the item
    ratings_from_trusted = R.loc[trusted_users, item].dropna()
    if not ratings_from_trusted.empty:
        # Simple average of trusted users' ratings (weights = trust strength)
        return float(ratings_from_trusted.mean())

    # Fallback to target user's average rating
    ua = user_avg.get(user, np.nan)
    if not np.isnan(ua):
        return float(ua)

    # Final fallback to global average
    return float(global_avg)

In [5]:
def recommend_items(user, R, T, user_avg, global_avg, top_n=3):
    # Items the user has not rated yet
    unrated_items = R.loc[user][R.loc[user].isna()].index.tolist()
    preds = []
    for it in unrated_items:
        p = predict_rating(user, it, R, T, user_avg, global_avg)
        preds.append((it, p))
    preds_sorted = sorted(preds, key=lambda x: x[1], reverse=True)
    return preds_sorted[:top_n]


In [6]:
# Example: predict some ratings and recommend for a few users
for u in users:
    recs = recommend_items(u, R, T, user_avg, global_avg, top_n=3)
    print(f"User {u} recommendations (item, predicted_rating): {recs}")

# Example: predict one specific (user,item)
print("\nSample predictions:")
print("Predict rating for user 0 on item 2 ->", predict_rating(0, 2, R, T, user_avg, global_avg))
print("Predict rating for user 2 on item 0 ->", predict_rating(2, 0, R, T, user_avg, global_avg))

User 0 recommendations (item, predicted_rating): [(3, 4.5), (4, 4.5), (2, 3.0)]
User 1 recommendations (item, predicted_rating): [(1, 5.0), (3, 4.0), (4, 4.0)]
User 2 recommendations (item, predicted_rating): [(1, 4.0), (0, 2.0), (3, 2.0)]
User 3 recommendations (item, predicted_rating): [(0, 4.0), (2, 4.0), (3, 4.0)]
User 4 recommendations (item, predicted_rating): [(0, 5.0), (1, 3.0), (2, 3.0)]
User 5 recommendations (item, predicted_rating): [(2, 5.0), (3, 5.0), (4, 5.0)]
User 6 recommendations (item, predicted_rating): [(0, 4.0), (4, 4.0), (2, 3.0)]
User 7 recommendations (item, predicted_rating): [(4, 5.0), (0, 4.0), (1, 2.5)]
User 8 recommendations (item, predicted_rating): [(2, 5.0), (3, 4.5), (1, 4.0)]
User 9 recommendations (item, predicted_rating): [(0, 4.5), (3, 4.5), (4, 4.5)]

Sample predictions:
Predict rating for user 0 on item 2 -> 3.0
Predict rating for user 2 on item 0 -> 2.0


In [7]:
# Quick leave-one-out MAE using trust-aware predictor:
errors = []
for idx, row in df_ratings.iterrows():
    u, it, true_r = row['user'], row['item'], row['rating']
    # Temporarily remove that rating
    orig = R.at[u, it]
    R.at[u, it] = np.nan
    # recompute user avg (for this temp state)
    temp_user_avg = R.mean(axis=1, skipna=True)
    pred = predict_rating(u, it, R, T, temp_user_avg, global_avg)
    errors.append(abs(pred - true_r))
    # put it back
    R.at[u, it] = orig

mae = float(np.mean(errors))
print("Leave-one-out MAE (trust-aware):", mae)

Leave-one-out MAE (trust-aware): 1.0859375
